# **Introdução ao Sktime — Uma Ação**

**Autor**: Jorge H. N. Viana

**Resumo**: Este notebook aplica modelos de séries temporais e de aprendizado de
máquina à previsão de retornos semanais da ação AGRO3.SA, utilizando o pacote
`sktime`, uma biblioteca especializada em *forecasting* com interface unificada
compatível com `scikit-learn`.

Três famílias de modelos são comparadas:
1. **SARIMAX**: modelo clássico ARIMA sazonal, com e sem variáveis exógenas (SELIC);
2. **GJR-GARCH**: modelo de volatilidade condicional com efeito alavancagem,
   estimado via simulação;
3. **Gradient Boosting (GBM)**: modelo de ML adaptado a séries temporais via
   `make_reduction`, que transforma o problema de previsão em regressão supervisionada.

Todos os modelos passam por otimização de hiperparâmetros via
`ForecastingRandomizedSearchCV` com validação cruzada em janelas deslizantes
(`SlidingWindowSplitter`). Ao final, os modelos ótimos são comparados no
conjunto de teste com horizonte de 24 semanas.

# 1. CARREGAMENTO DOS PACOTES

Nesta seção são importadas as bibliotecas necessárias. Destacam-se:
- `sktime`: pacote central de *forecasting*, qque oferece `SARIMAX`, `ARCH`
  (GARCH/GJR-GARCH), `SlidingWindowSplitter`, `ForecastingRandomizedSearchCV`
  e `make_reduction` (adaptação de modelos do scikit-learn para séries temporais);
- `yfinance`: coleta de preços do Yahoo Finance;
- `scikit-learn`: `GradientBoostingRegressor` como estimador base para o GBM;
- `scipy.stats`: distribuições para os espaços de busca de hiperparâmetros;
- `statsmodels`: usado internamente pelo `sktime` para estimação do SARIMA.

In [ ]:
# Pacotes
import numpy as np  
import os as os      
import pandas as pd           
import yfinance as yf  
from datetime import date, timedelta
from pathlib import Path
from multiprocessing import cpu_count
import warnings
import requests
import io
from itertools import product
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

from sktime.split import temporal_train_test_split, SlidingWindowSplitter
from sktime.forecasting.model_evaluation import evaluate
from sktime.forecasting.model_selection import ForecastingRandomizedSearchCV
from sktime.forecasting.sarimax import SARIMAX
from sktime.forecasting.arch import ARCH
from sktime.forecasting.compose import make_reduction
from sklearn.ensemble import GradientBoostingRegressor as GBM

from scipy.stats import randint, loguniform, triang, geom

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.filterwarnings(
    action='ignore',
    category=UserWarning,
    module='statsmodels.*'  
)

# 2. CONFIGURAÇÕES

Definem-se os parâmetros globais do notebook:
- **Período de análise**: 5 anos de dados (~260 semanas de observações),
  período mais longo que os notebooks anteriores para capturar ciclos completos;
- **Diretórios**: *Input*, *Stage*, *Output* e *Figures*;
- **Número de jobs**: `cpu_count() - 2` para paralelização do grid search;
- **Parâmetros da janela deslizante**:
  - `wl = 104` (2 anos): tamanho da janela de treino;
  - `sw = 12` (trimestral): passo de avanço entre janelas;
  - `fh = range(1, 25)`: horizonte de previsão de até 24 semanas (~6 meses).

In [3]:
# Define o período de tempo para coleta dos dados
# fim    = date.today()
fim    = date(2025, 12, 31)
inicio = fim - timedelta(days=365*5)

# Diretórios
dir_raiz   = "G:\\Meu Drive\\Ensino\\TAF\\taf-code"
input_dir  = Path(dir_raiz, 'Data\\Input\\')
stage_dir  = Path(dir_raiz, 'Data\\Stage\\')
output_dir = Path(dir_raiz, 'Data\\Output\\')
figure_dir = Path(dir_raiz, 'Figures\\')

# Jobs
jobs = cpu_count() - 2

# Windows
wl  = 104 # window length
sw  = 12 # step window
fh = range(1, 25)

# 3. COLETA E PREPARAÇÃO DOS DADOS

Nesta seção obtêm-se os dados brutos dos preços da ação AGRO3.SA e da taxa SELIC
e realiza-se o pré-processamento necessário: log-retornos semanais, conversão
da SELIC para periodicidade semanal e criação de variáveis defasadas (*lags*).
As defasagens da SELIC serão usadas como variáveis exógenas nos modelos SARIMAX e GARCH.

### 3.1 Coleta dos Preços (AGRO3.SA)

Os preços de fechamento ajustados (*Adj Close*) da ação AGRO3.SA são baixados
do Yahoo Finance para um período de 5 anos. O DataFrame resultante tem datas como
índice e ticker como coluna única.

In [4]:
# Define os ativos a serem utilizados
ativos = ['AGRO3.SA']  

# Coleta os dados do Yahoo Finance
precos = (yf
         .download(ativos, start=inicio, end=fim, 
                   auto_adjust=False)
         ['Adj Close']
         )
precos.head()

[*********************100%***********************]  1 of 1 completed


Ticker,AGRO3.SA
Date,
2021-01-04,15.531752
2021-01-05,15.966640
2021-01-06,16.165447
2021-01-07,16.463657
2021-01-08,16.090893


### 3.2 Taxa SELIC

A taxa SELIC diária é obtida via API do Banco Central (código 1178). Os dados
vêm no formato CSV brasileiro (separador `;`, vírgula decimal) e são convertidos
para `float`. São coletadas 1.254 observações diárias ao longo dos 5 anos do período.

In [5]:
api_url = ('https://api.bcb.gov.br/dados/serie/bcdata.sgs.1178/dados?formato=csv&dataInicial={}&dataFinal={}'
           .format(inicio.strftime('%d/%m/%Y'), fim.strftime('%d/%m/%Y')))

selic    = requests.get(api_url)
selic_io = io.StringIO(selic.text)
selic    = (pd.read_csv(selic_io, sep=';', decimal=',') 
            .rename(columns={'valor': 'selic', 'data': 'Date'})
            .assign(Date = lambda x: pd.to_datetime(x['Date'], dayfirst=True))
            .astype({'selic': float})
            )

selic.info()
selic.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1256 entries, 0 to 1255
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    1256 non-null   datetime64[ns]
 1   selic   1256 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 19.8 KB


,Date,selic
0,2021-01-04,1.9
1,2021-01-05,1.9
2,2021-01-06,1.9
3,2021-01-07,1.9
4,2021-01-08,1.9
5,2021-01-11,1.9
6,2021-01-12,1.9
7,2021-01-13,1.9
8,2021-01-14,1.9
9,2021-01-15,1.9


### 3.3 Tratamento

Os dados brutos são transformados em séries utilizáveis pelos modelos de
previsão. O tratamento segue três etapas: cálculo dos log-retornos semanais
(target), conversão da SELIC para frequência semanal e consolidação das
variáveis defasadas que servirão como exógenas nos modelos SARIMAX e GJR-GARCH.

#### 3.3.1 Log-Retornos Semanais

Calculam-se os log-retornos semanais pela diferença de logaritmos dos preços
reamostrados para o fechamento de cada domingo (`.resample('W').last()`).
O resultado é transformado para o formato longo com `melt()`, permitindo
consistência com análises multi-ativos futuras (embora aqui haja apenas AGRO3).

In [6]:
log_retornos = ((np.log(precos.resample('W').last()) 
                 - np.log(precos.resample('W').last().shift(1))                
                 )
                .dropna()
                .reset_index() 
                .rename_axis(None, axis = 1)
                .rename(columns=lambda x: x.replace('.SA', ''))
                .melt(id_vars = 'Date', var_name = 'ativo', 
                      value_name = 'retorno')
                .assign(ativo = lambda x: x['ativo'].str.replace('.SA', '')
                        )
                )

log_retornos.head()

,Date,ativo,retorno
0,2021-01-17,AGRO3,0.041594
1,2021-01-24,AGRO3,-0.022852
2,2021-01-31,AGRO3,-0.023386
3,2021-02-07,AGRO3,-0.080754
4,2021-02-14,AGRO3,-0.033351


#### 3.3.2 SELIC Semanal

A SELIC diária é convertida para taxa semanal via juros compostos:

$$ R_{\text{semanal}} = (1 + R_{\text{diária}})^{1/52} - 1 $$

A reamostragem semanal usa o último valor disponível em cada semana. Nos
períodos em que a SELIC permaneceu constante (ex.: 3,65% entre abril e
maio de 2020), a variável `selic_w` não contribui com variabilidade preditiva.

In [7]:
selic = (selic
         .assign(selic_w = lambda x: (1 + x["selic"])**(1/52) - 1)
         .set_index('Date')
         .resample('W')
         .last()
         .reset_index()
         )

selic.head(10)

,Date,selic,selic_w
0,2021-01-10,1.9,0.020686
1,2021-01-17,1.9,0.020686
2,2021-01-24,1.9,0.020686
3,2021-01-31,1.9,0.020686
4,2021-02-07,1.9,0.020686
5,2021-02-14,1.9,0.020686
6,2021-02-21,1.9,0.020686
7,2021-02-28,1.9,0.020686
8,2021-03-07,1.9,0.020686
9,2021-03-14,1.9,0.020686


#### 3.3.3 Consolidação dos Dados

Consolidam-se os log-retornos com a SELIC semanal e criam-se 6 variáveis
defasadas (*lags* de 1 a 3 semanas para retorno e SELIC) usando
`groupby('ativo').shift()`. Linhas com `NaN` (início da série) são removidas.

> **Nota**: Diferentemente do notebook de Scikit-Learn, apenas as defasagens
> da SELIC serão usadas como exógenas (X). As defasagens do retorno são mantidas
> no DataFrame mas não entram nos modelos, pois o SARIMAX já modela a estrutura de
> autocorrelação internamente.

In [8]:
dados = (log_retornos
.merge(selic[['Date', 'selic_w']], 
on = 'Date', how = 'left')
.sort_values(['ativo', 'Date'])
.assign(retorno_lag_1 = lambda x: x.groupby('ativo')['retorno'].shift(1),
        retorno_lag_2 = lambda x: x.groupby('ativo')['retorno'].shift(2),
        retorno_lag_3 = lambda x: x.groupby('ativo')['retorno'].shift(3),
        selic_w_lag_1 = lambda x: x['selic_w'].shift(1),
        selic_w_lag_2 = lambda x: x['selic_w'].shift(2),
        selic_w_lag_3 = lambda x: x['selic_w'].shift(3)
        )
.dropna()
.reset_index(drop = True)
)

dados.head()

,Date,ativo,retorno,selic_w,retorno_lag_1,retorno_lag_2,retorno_lag_3,selic_w_lag_1,selic_w_lag_2,selic_w_lag_3
0,2021-02-07,AGRO3,-0.080754,0.020686,-0.023386,-0.022852,0.041594,0.020686,0.020686,0.020686
1,2021-02-14,AGRO3,-0.033351,0.020686,-0.080754,-0.023386,-0.022852,0.020686,0.020686,0.020686
2,2021-02-21,AGRO3,-0.015334,0.020686,-0.033351,-0.080754,-0.023386,0.020686,0.020686,0.020686
3,2021-02-28,AGRO3,-0.028663,0.020686,-0.015334,-0.033351,-0.080754,0.020686,0.020686,0.020686
4,2021-03-07,AGRO3,0.059100,0.020686,-0.028663,-0.015334,-0.033351,0.020686,0.020686,0.020686


# 4. DIVISÃO DOS DADOS

A divisão treino-teste é feita com `temporal_train_test_split` do `sktime`,
que garante que o conjunto de treino contenha apenas observações anteriores
às do teste, algo essencial para séries temporais. O conjunto de teste corresponde
ao último terço da amostra (~86 semanas).

> **Importante**: Diferentemente do `train_test_split` do scikit-learn, o
> `temporal_train_test_split` do sktime preserva o índice temporal **necessariemente** e não
> oferece o parâmetro `shuffle`.

### 4.1 Separação em Features (X) e Target (y)

O target (`y`) é o log-retorno semanal da AGRO3. As *features* exógenas (`X`)
incluem apenas a SELIC semanal corrente e suas defasagens (4 variáveis no total).

As defasagens do retorno não são incluídas em X porque os modelos SARIMA e
GARCH já modelam a estrutura de autocorrelação dos retornos internamente.
Incluí-las como exógenas poderia causar redundância e instabilidade numérica.

In [9]:
y = dados['retorno']
X = dados[['selic_w', 'selic_w_lag_1', 'selic_w_lag_2', 
           'selic_w_lag_3']]
X.head()

,selic_w,selic_w_lag_1,selic_w_lag_2,selic_w_lag_3
0,0.020686,0.020686,0.020686,0.020686
1,0.020686,0.020686,0.020686,0.020686
2,0.020686,0.020686,0.020686,0.020686
3,0.020686,0.020686,0.020686,0.020686
4,0.020686,0.020686,0.020686,0.020686


### 4.2 Divisão Treino-Teste

Utiliza-se `temporal_train_test_split` com `test_size=1/3`. O conjunto de
treino resultante tem 172 observações semanais (~3,3 anos), e o teste tem
86 semanas (~1,7 anos). A ordem cronológica é estritamente respeitada.

In [10]:
X_train, X_test, y_train, y_test = temporal_train_test_split(X, y, test_size=1/3)
y_train.info()

<class 'pandas.core.series.Series'>
Index: 171 entries, 0 to 170
Series name: retorno
Non-Null Count  Dtype  
--------------  -----  
171 non-null    float64
dtypes: float64(1)
memory usage: 2.7 KB


# 5. APLICAÇÃO — MODELO SARIMA

O **SARIMA** (*Seasonal ARIMA*) é a extensão sazonal do modelo ARIMA clássico.
Sua especificação geral é SARIMA(p,d,q)(P,D,Q)[s], onde:
- **Ordem não sazonal** (p,d,q): parte AR(p), diferenciação(d), MA(q);
- **Ordem sazonal** (P,D,Q)[s]: análoga, aplicada a defasagens sazonais s;
- No contexto de retornos semanais, usa-se s=52 para capturar sazonalidade anual.

O `sktime` encapsula a estimação via `statsmodels` e oferece interface compatível
com scikit-learn (`fit`, `predict`). A otimização de hiperparâmetros é feita em
4 etapas progressivas: do modelo mais simples ao mais completo.

### 5.1 Treinamento e Avaliação sem Validação Cruzada

Treina-se um SARIMA com hiperparâmetros padrão do sktime (`maxiter=500`) nos
dados de treino e avalia-se no conjunto de teste completo (86 semanas).
As métricas reportadas são MAPE (*Mean Absolute Percentage Error*) e RMSE.

Embora simples, esta abordagem é sensível à partição específica. A validação
cruzada nas próximas seções produzirá estimativas mais robustas do erro.

In [13]:
# Instanciação do Modleo
model_sarima  = SARIMAX(random_state=42, 
                        maxiter=500,
                        disp=False
                        )

model_sarima.fit(y_train)

y_pred_sarima = model_sarima.predict(
    fh = range(1, len(y_test) + 1))

mape_sarima = mean_absolute_percentage_error(y_test,
                                        y_pred_sarima)
rmse_sarima = root_mean_squared_error(y_test, 
                                       y_pred_sarima)

print(f"MAPE of SARIMA Model: {mape_sarima:.4f}")
print(f"RMSE of Bagging Model: {rmse_sarima:.4f}")

MAPE of SARIMA Model: 105152412897.6127
RMSE of Bagging Model: 0.0240


### 5.2 Treinamento e Avaliação com Validação Cruzada

Utiliza-se `SlidingWindowSplitter` com janela de 104 semanas e passo de 12
semanas para gerar múltiplos cenários de avaliação. A função `evaluate()`
do sktime automatiza o processo: para cada janela, re-treina o modelo e
avalia no horizonte de previsão `fh`.

Com 172 observações de treino, janela de 104 e passo 12, obtêm-se 4 avaliações
independentes. O MAPE médio reportado é a média dos MAPEs de cada corte.

> **Nota sobre `strategy="refit"`**: O modelo é re-estimado em cada janela,
> simulando o cenário real onde o modelo seria atualizado periodicamente.
> A alternativa `strategy="update"`, se disponível, apenas atualizaria os parâmetros, sendo
> computacionalmente mais barata mas menos realista para mudanças estruturais.

In [16]:
# Instanciação das Janelas
spliter = SlidingWindowSplitter(fh= fh,
            window_length= wl,
            step_length = sw,
            )

# reinstanciando o modelo
model_sarima  = SARIMAX(random_state=42, 
                        maxiter=500
                        )

# Validação cruzada
cv_sarima = evaluate( 
              model_sarima,
              y = y_train, 
              cv = spliter, 
              scoring = mean_absolute_percentage_error,
              strategy="refit"
            )

print(f"Mean MAPE: {cv_sarima.iloc[:,0].mean():.4f}")
print(f"Mean RMSE: {cv_sarima.iloc[:,1].mean():.4f}")
cv_sarima.head()

Mean MAPE: 1.0256
Mean RMSE: 0.0384


,test__DynamicForecastingErrorMetric,fit_time,pred_time,len_train_window,cutoff
0,1.084475,0.043259,0.004334,104,103
1,1.023498,0.031909,0.004345,104,115
2,0.963424,0.058913,0.004771,104,127
3,1.031132,0.019323,0.004030,104,139


### 5.3 Otimização com Grid Search

Otimizam-se os hiperparâmetros do SARIMA via `ForecastingRandomizedSearchCV`,
o equivalente do `RandomizedSearchCV` para modelos de previsão. O espaço de busca
inclui:
- `order` (p,d,q): combinações de p ∈ {1,2}, d=0, q ∈ {0,1,2};
- `seasonal_order` (P,D,Q,s): combinações sazonais com s=52;
- `trend`: tipos de tendência determinística (n, c, t, ct).

O total de combinações é 120 (6 × 4 × 5 com o seasonal (0,0,0,0) incluso).
A busca é executada com `backend="loky"` para paralelização. O RMSE é usado
como métrica de *scoring* (quanto menor, melhor).

In [18]:
# Reinstanciando o modelo
model_sarima  = SARIMAX(random_state=42, 
                        maxiter=500
                        )

# Espaço de busca para os hiperparâmetros
## Range de cada termo das ordens
s       = 52
p_range = range(1, 3) 
d_range = range(0, 1)  
q_range = range(0, 3)  
P_range = range(0, 2)  
D_range = range(0, 1)  
Q_range = range(0, 2) 

## Combinações
orders   = list(product(p_range, d_range, q_range))
s_orders = [(P, D, Q, s) for P, D, Q in 
            product(P_range, D_range, Q_range)]
s_orders.append((0,0,0,0))

param_distributions = {
    "order": orders,
    "seasonal_order": s_orders,
    "trend": ["n", "c", "t", "ct"]
}

# Instância da procura
random_search = ForecastingRandomizedSearchCV(
    forecaster=model_sarima,        
    cv=spliter,                           
    param_distributions=param_distributions, 
    n_iter=100,                            
    scoring=root_mean_squared_error,               
    random_state=42,      
    strategy="refit",
    backend = "loky",
    backend_params = {
        "n_jobs": jobs
    }            
)

# Executando a busca
random_search.fit(y_train)

# Melhor conjunto de hiperparâmetros
best_params = random_search.best_params_
best_score  = random_search.best_score_
sarima_opt  = random_search.best_forecaster_
print(f"Best Param: {best_params}")
print(f"Optimized RMSE: {best_score:.4f}")

Best Param: {'trend': 'n', 'seasonal_order': (0, 0, 0, 52), 'order': (1, 0, 0)}
Optimized RMSE: 0.0342


### 5.4 Adição de Variáveis Exógenas (SARIMAX)

O mesmo grid search é repetido incluindo as variáveis exógenas (SELIC e suas
defasagens) via `fit(y_train, X=X_train)`. O modelo passa a ser um **SARIMAX** —
SARIMA com variáveis eXógenas.

A adição de exógenas pode melhorar a precisão se a SELIC contiver informação
preditiva sobre os retornos (ex.: relação entre política monetária e valuations).
O RMSE reportado permite comparar diretamente: se o SARIMAX tiver RMSE menor
que o SARIMA simples, as exógenas agregam valor preditivo.

In [19]:
# Reexecutando a busca
random_search.fit(y_train, X = X_train)

# # Melhor conjunto de hiperparâmetros
best_params = random_search.best_params_
best_score  = random_search.best_score_
sarimax_opt  = random_search.best_forecaster_
print(f"Best Param: {best_params}")
print(f"Optimized RMSE: {best_score:.4f}")

Best Param: {'trend': 'ct', 'seasonal_order': (0, 0, 1, 52), 'order': (1, 0, 2)}
Optimized RMSE: 0.0342


# 6. APLICAÇÃO — MODELO GJR-GARCH

O **GJR-GARCH** (Glosten, Jagannathan & Runkle, 1993) é um modelo de volatilidade
condicional que captura o **efeito alavancagem**: choques negativos tendem a
aumentar mais a volatilidade do que choques positivos de mesma magnitude.

No `sktime`, o modelo é implementado na classe `ARCH`, que suporta as variantes
GARCH, EGARCH e GJR-GARCH conforme os parâmetros `p` (ARCH), `o` (alavancagem)
e `q` (GARCH). A estimação é feita via simulação com 100 trajetórias.

O espaço de busca inclui:
- `p` (1–4): termos ARCH; `o` (0–2): termos de alavancagem (o ≥ 1 ativa GJR);
  `q` (0–4): termos GARCH;
- `dist`: distribuição dos erros (Normal, t-Student, t assimétrica);
- `mean`: especificação da média (Constante, Zero, AR, ARX);
- `lags` (1–2): defasagens para a equação da média.

O modelo é estimado com exógenas (`X=X_train`), permitindo que a SELIC afete
a equação da média dos retornos, enquanto a volatilidade segue o processo GJR-GARCH.

In [20]:
# Instanciação do modelo
model_arch  = ARCH(method = "simulation",
                   simulations = 100,
                   random_state = 42
                   )

# Espaço de busca para os hiperparâmetros    
param_arch = {
    "p": randint(1, 5),        
    "o": randint(0, 3),
    "q": randint(0, 5),  
    "dist": ['normal', 't', 'skewt'],
    "mean": ['Constant', 'Zero', 'AR',
             'ARX'],
    "lags": randint(1, 3),    
}

# Instância da procura
random_arch = ForecastingRandomizedSearchCV(
    forecaster=model_arch,        
    cv=spliter,                           
    param_distributions=param_arch, 
    n_iter=1000,                            
    scoring=root_mean_squared_error,               
    random_state=42,      
    strategy="refit",
    backend = "loky",
    backend_params = {
        "n_jobs": jobs
    }            
)

# Executando a busca
random_arch.fit(y_train, X = X_train)

# # Melhor conjunto de hiperparâmetros
best_params_arch = random_arch.best_params_
best_score_arch  = random_arch.best_score_
arch_opt         = random_arch.best_forecaster_
print(f"Best Param: {best_params_arch}")
print(f"Optimized RMSE: {best_score_arch:.4f}")


Best Param: {'dist': 'normal', 'lags': 1, 'mean': 'Zero', 'o': 2, 'p': 3, 'q': 2}
Optimized RMSE: 0.0341


# 7. APLICAÇÃO — GRADIENT BOOSTING (GBM)

O **Gradient Boosting** é adaptado a séries temporais via `make_reduction`, que
transforma o problema de previsão em um problema de regressão supervisionada:
cria variáveis defasadas automaticamente a partir da série temporal e treina
um regressor do scikit-learn sobre elas.

Os parâmetros configurados são:
- `window_length=52`: o modelo "enxerga" 1 ano de histórico (52 semanas) para
  fazer cada previsão;
- `strategy="recursive"`: para horizontes de múltiplos passos (fh), o modelo
  usa suas próprias previsões como entrada para os passos seguintes.

O espaço de busca do GBM inclui `n_estimators` (distribuição geométrica),
`learning_rate` (log-uniforme entre 0,001 e 0,5), `max_depth`, `min_samples_leaf`
e `max_features`.

> **Comparação esperada**: Em geral, o GBM tende a superar modelos lineares
> como SARIMA em problemas com não linearidades. No entanto, a baixa razão
> sinal-ruído dos retornos financeiros limita o ganho potencial de qualquer modelo.

In [21]:
# Instaciando o Estimador de GBM
estimator_gbm  = GBM(random_state=42)
model_gbm = make_reduction(estimator_gbm, 
                           window_length=52, 
                           strategy="recursive")

# Espaço de busca
param_gbm = {            
'n_estimators': geom(p=0.04, loc=9), 
'max_features':triang(c=0, loc=0.5, 
                      scale=0.5),
'max_depth': randint(3, 15),
'min_samples_leaf': randint(1, 6),
'learning_rate': loguniform(0.001,
                             0.5)
}

# Instância da procura
random_gbm = ForecastingRandomizedSearchCV(
    forecaster=model_gbm,        
    cv=spliter,                           
    param_distributions=param_gbm, 
    n_iter=1000,                            
    scoring=root_mean_squared_error,               
    random_state=42,      
    strategy="refit",
    backend = "loky",
    backend_params = {
        "n_jobs": jobs
    }            
)

# Executando a busca
random_gbm.fit(y_train)

# Melhor conjunto de hiperparâmetros
best_params_gbm = random_gbm.best_params_
best_score_gbm  = random_gbm.best_score_
gbm_opt         = random_gbm.best_forecaster_
print(f"Best Param: {best_params_gbm}")
print(f"Optimized RMSE: {best_score_gbm:.4f}")

Best Param: {'learning_rate': np.float64(0.04466621498687272), 'max_depth': 13, 'max_features': np.float64(0.5175701880881012), 'min_samples_leaf': 5, 'n_estimators': 22}
Optimized RMSE: 0.0320


# 8. AVALIAÇÃO DOS MODELOS NO CONJUNTO DE TESTE

Nesta seção final, os modelos ótimos (selecionados via grid search) são
avaliados no conjunto de teste, dados que **nenhum modelo viu durante o
treino ou a otimização**. Esta é a métrica mais honesta de desempenho.

São comparados:
- **GBM ótimo** vs. **SARIMA ótimo** (sem exógenas) — ambos otimizados via
  `ForecastingRandomizedSearchCV`;
- Horizonte de previsão: 24 semanas à frente.

### 8.1 Avaliação no Teste

Cada modelo ótimo gera previsões para o horizonte completo do conjunto de teste
(`len(y_test) ≈ 86 semanas`) e o RMSE é calculado comparando com os valores
reais observados.

> **Interpretação**: O RMSE no teste é tipicamente maior que o RMSE da validação
> cruzada, pois reflete o desempenho em dados completamente não vistos. Se a
> diferença for muito grande, pode indicar *overfitting* durante a otimização.

As previsões do conjunto de teste também podem ser inspecionadas descritivamente
(média, desvio-padrão, mínimo e máximo) para verificar se os valores previstos
são plausíveis dentro da escala histórica dos retornos.

In [22]:
y_pred    = gbm_opt.predict(fh=range(1, len(y_test) + 1))
rmse_test = root_mean_squared_error(y_test, y_pred)
print(f"RMSE no Teste: {rmse_test:.4f}")

RMSE no Teste: 0.0251


In [23]:
y_pred = sarima_opt.predict(fh=range(1, len(y_test) + 1))
rmse_test = root_mean_squared_error(y_test, y_pred)
print(f"RMSE no Teste: {rmse_test:.4f}")

RMSE no Teste: 0.0238


### 8.2 Previsões para o Futuro

Utilizando os modelos ótimos, geram-se previsões *fora da amostra* para
horizontes futuros. No código, os três modelos são usados para prever 24
semanas à frente a partir do último ponto conhecido.

As previsões são concatenadas em um DataFrame comparativo, permitindo
visualizar divergências entre as previsões dos diferentes modelos. Divergências
grandes indicam incerteza sobre a direção futura e podem ser usadas como sinal
de cautela na tomada de decisão.

> **Cuidado**: Previsões de retornos financeiros têm incerteza inerentemente
> alta. Mesmo o melhor modelo pode ter RMSE da mesma ordem de grandeza que o
> desvio-padrão incondicional dos retornos, ou seja, a previsão pontual pode
> não ser melhor que a média histórica.

In [32]:
# Vetor de dados recentes
y_recent = y.tail(wl)

# Atualização do forecaster GBM
gbm_opt.fit(y_recent)

# Previsões do Futuro (fh = 24)
gbm_predictions = gbm_opt.predict(fh=fh)

# Atualização do forecaster SARIMA
sarima_opt.fit(y_recent)

# Previsões do Futuro (fh = 24)
sarima_predictions = sarima_opt.predict(fh=fh)

# DataFrame consolidado com previsões de ambos os modelos
data_inicio = dados["Date"].iloc[-1]

datas = pd.date_range(start=data_inicio,
                      periods=len(gbm_predictions),
                      freq='W-SUN')

predictions = pd.DataFrame({
    "Date": datas,
    "GBM": gbm_predictions.values.flatten() if hasattr(gbm_predictions, 'values') else gbm_predictions,
    "SARIMA": sarima_predictions.values.flatten() if hasattr(sarima_predictions, 'values') else sarima_predictions
})

print("\n=== Comparação GBM vs SARIMA ===")
print(predictions[["GBM", "SARIMA"]].describe())


=== Comparação GBM vs SARIMA ===
             GBM        SARIMA
count  24.000000  2.400000e+01
mean    0.000450  4.937189e-05
std     0.005827  3.058488e-04
min    -0.006957 -3.231733e-04
25%    -0.004301 -2.824417e-11
50%     0.000615  2.590948e-18
75%     0.003251  1.266522e-10
max     0.016042  1.449171e-03


# 9. LIMIETAÇÕES E PRÓXIMOS PASSOS

## 9.1 Limitações

- Apenas uma ação (AGRO3) foi analisada — os resultados podem não generalizar
  para outros setores ou perfis de risco;
- O período de 5 anos (2020–2025) inclui a pandemia de COVID-19, que gerou
  volatilidade atípica — modelos treinados nesse período podem não performar
  bem em regimes de baixa volatilidade;
- A SELIC variou pouco em vários subperíodos, reduzindo sua utilidade como
  variável exógena;
- O horizonte de previsão de 24 semanas é relativamente curto para capturar
  ciclos de negócios completos.

## 9.2 Extensões sugeridas

1. Expandir a análise para múltiplas ações, comparando a performance relativa
   dos modelos entre setores (ex.: commodities vs. financeiro vs. utilities);
2. Testar outros modelos disponíveis no `sktime`: ExponentialSmoothing, TBATS,
   Prophet, redes neurais (LSTM via `make_reduction`);
3. Implementar *backtesting* com re-treino periódico (*walk-forward validation*),
   simulando o uso do modelo em produção ao longo de todo o período;
4. Adicionar variáveis exógenas macroeconômicas: IPCA, PIB, câmbio, EMBI+,
   que podem capturar dimensões de risco não refletidas na SELIC;
5. Avaliar os modelos não apenas por métricas de erro (RMSE, MAPE), mas
   também por métricas financeiras de uma estratégia baseada nas previsões.
6. Comparar com *benchmarks* ingênuos: média histórica, *random walk* e
   modelo AR(1), se o RMSE do modelo não for significativamente inferior,
   a complexidade adicional pode não se justificar.